# Data Preprocessing

In [1]:
import pandas as pd

## Data Checks

In [2]:
# load data
ipi_raw = pd.read_csv("data/ipi_2d.csv", parse_dates = ["date"], dtype={"division": str}) # division kept as string so 06 doesnt become 6
print(ipi_raw.shape) # (10660, 4)

# Working raw dataset
ipi = ipi_raw.copy()

ipi.describe()

# Range of date
print("\nRange of Dates:\n", ipi["date"].min(), ipi["date"].max()) # 1 Jan 2015 to 1 September 2026

# Range of index grouped by series
print("\nRange of index grouped by series:\n", ipi.groupby("series")["index"].agg(["min","max","mean"]))

# Range of dates by series
print("\nRange of dates grouped by series:\n", ipi.groupby("series")["date"].agg(["min","max"])) # Dates for abs and growth_yoy same (jan missing for growth_mom)

# Number of unique sectors
print("\nNumber of unique divisions:\n", ipi["division"].nunique()) # 26 sectors

print("\nUnique divisions:\n", ipi["division"].unique()) # sectors 6, 10 - 33, 35

# Check for duplicated values
print("\nNumber of duplicated values:\n", ipi.duplicated(subset=["series","date","division"]).sum()) # 0 duplicates

# Months per sector
ipi.groupby(["series", "division"]).size().unique()

# Check for missing values
ipi.isna().sum() # no missing values (rows missing; will show up in wide ipi)



(10660, 4)

Range of Dates:
 2015-01-01 00:00:00 2026-09-01 00:00:00

Range of index grouped by series:
                min       max        mean
series                                  
abs          2.384   264.512  122.591121
growth_mom -97.564  2385.057    2.093495
growth_yoy -97.964  3960.100    8.735070

Range of dates grouped by series:
                   min        max
series                          
abs        2015-01-01 2026-09-01
growth_mom 2015-02-01 2026-09-01
growth_yoy 2016-01-01 2026-09-01

Number of unique divisions:
 26

Unique divisions:
 <StringArray>
['06', '10', '11', '12', '13', '14', '15', '16', '17', '18', '31', '19', '20',
 '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '32', '33', '35']
Length: 26, dtype: str

Number of duplicated values:
 0


series      0
date        0
division    0
index       0
dtype: int64

## Reshape Data

In [3]:
ipi_wide = (ipi.pivot_table(index = ["division", "date"],
                        columns = "series",
                        values = "index")
            .reset_index()
            .sort_values(["division", "date"])
)


print(ipi_wide.shape)               # (3666, 5)
print(ipi_wide.head())
print(ipi_wide.isna().sum())        # growth_mom: 26, growth_yoy: 312

# Keep manufacturing sectors only
ipi_wide = ipi_wide[ipi_wide["division"].between("10", "33")].copy()

ipi_wide[ipi_wide["growth_yoy"].isna()].groupby("division")["date"].agg(["min", "max", "count"])
# 12 NA per sector from Jan 2015 to Sept 2015

# Save ipi_wide as csv
ipi_wide.to_csv("data/ipi_wide.csv", index = False)

(3666, 5)
series division       date      abs  growth_mom  growth_yoy
0            06 2015-01-01  107.236         NaN         NaN
1            06 2015-02-01   96.262     -10.233         NaN
2            06 2015-03-01  107.744      11.928         NaN
3            06 2015-04-01   96.051     -10.853         NaN
4            06 2015-05-01  103.504       7.760         NaN
series
division        0
date            0
abs             0
growth_mom     26
growth_yoy    312
dtype: int64
